# 🧩 Mini-Lab: Retry with Backoff

**Module 10: Production Deployment** | **Type: Mini-Lab (Brick)**

---

## Learning Objectives

By the end of this mini-lab, you will be able to:

1. **Understand** why transient failures require retry logic instead of immediate failure
2. **Implement** exponential backoff with jitter for retrying failed LLM API calls
3. **Recognize** how fallback patterns provide graceful degradation when all retries are exhausted
4. **Compare** naive retry vs. exponential backoff vs. fallback strategies

## Target Concepts

| Concept | Description |
|---------|-------------|
| Retry Patterns | Automatically re-attempting failed API calls with exponential backoff and jitter to handle transient errors without overwhelming the server |
| Fallback Patterns | Providing a graceful degradation path (e.g., a cheaper model or a canned response) when all retry attempts are exhausted |

## Setup

In [3]:
import time
import random
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import display, Markdown

load_dotenv()

def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))

client = OpenAI()
MODEL = "gpt-4o-mini"
print("✅ Ready")

✅ Ready


## Why Retry?

LLM API calls can fail for **transient** reasons that have nothing to do with your code:

- **Rate limits** (429) — you sent too many requests
- **Server overload** (503) — the provider is temporarily busy
- **Network glitches** — a brief connectivity hiccup

These errors are *temporary*. If you simply wait and try again, the request usually succeeds. But **how** you wait matters:

| Strategy | Wait Pattern | Problem |
|----------|-------------|----------|
| No retry | Give up immediately | Unnecessary failure |
| Instant retry | 0s, 0s, 0s | Hammers the server, makes rate limits worse |
| Fixed delay | 2s, 2s, 2s | Better, but all clients retry at the same time |
| **Exponential backoff + jitter** | 1s, 2.3s, 5.1s | ✅ Spreads out retries, gives the server time to recover |

## Step 1 — Simulate Transient Failures

To test retry logic without burning real API credits on forced errors, we'll create a function that **simulates flaky behavior** — it fails a configurable number of times before succeeding.

In [4]:
class FlakyLLM:
    """
    Wraps the real OpenAI client but simulates transient failures.
    Fails `fail_count` times, then succeeds.
    """

    def __init__(self, real_client: OpenAI, fail_count: int = 2):
        self.real_client = real_client
        self.fail_count = fail_count
        self.attempts = 0

    def reset(self):
        self.attempts = 0

    def chat(self, message: str, temperature: float = 0) -> str:
        self.attempts += 1
        if self.attempts <= self.fail_count:
            raise Exception(f"Simulated transient error (attempt {self.attempts})")

        resp = self.real_client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": message}],
            temperature=temperature,
        )
        return resp.choices[0].message.content


flaky = FlakyLLM(client, fail_count=2)
print("✅ FlakyLLM ready — will fail 2 times, then succeed")

✅ FlakyLLM ready — will fail 2 times, then succeed


## Step 2 — No Retry (Baseline)

Without retry logic, the first transient error kills the request.

In [5]:
flaky.reset()

try:
    result = flaky.chat("What is exponential backoff? One sentence.")
    print(f"✅ Success: {result}")
except Exception as e:
    print(f"❌ Failed: {e}")
    print("   Without retries, the first transient error is fatal.")

❌ Failed: Simulated transient error (attempt 1)
   Without retries, the first transient error is fatal.


## Step 3 — Exponential Backoff with Jitter

Now let's build a proper retry function. The key formula:

```
wait_time = base_delay × (2 ^ attempt) + random_jitter
```

- **Exponential backoff**: each retry waits *longer* than the last (1s → 2s → 4s → ...)
- **Jitter**: a random addition so that multiple clients don't all retry at the exact same moment
- **Max delay cap**: prevents absurdly long waits

In [6]:
def retry_with_backoff(
    func,
    max_retries: int = 3,
    base_delay: float = 1.0,
    max_delay: float = 10.0,
):
    """
    Call `func()`. If it raises an exception, retry with exponential
    backoff + jitter up to `max_retries` times.

    Returns: (result, attempts_taken)
    Raises: the last exception if all retries fail.
    """
    last_exception = None

    for attempt in range(max_retries + 1):  # attempt 0 = first try
        try:
            result = func()
            return result, attempt + 1
        except Exception as e:
            last_exception = e
            if attempt < max_retries:
                # Calculate delay: exponential backoff + jitter
                delay = min(base_delay * (2 ** attempt), max_delay)
                jitter = random.uniform(0, delay * 0.5)
                wait = delay + jitter
                print(f"  ⚠️  Attempt {attempt + 1} failed: {e}")
                print(f"      Retrying in {wait:.2f}s (base={delay:.1f}s + jitter={jitter:.2f}s)")
                time.sleep(wait)
            else:
                print(f"  ⚠️  Attempt {attempt + 1} failed: {e}")

    raise last_exception


print("✅ retry_with_backoff() defined")

✅ retry_with_backoff() defined


### Try It — Retry Succeeds After Transient Failures

Our `FlakyLLM` fails twice then succeeds. With `max_retries=3`, the third attempt should work.

In [7]:
flaky.reset()

start = time.perf_counter()
result, attempts = retry_with_backoff(
    func=lambda: flaky.chat("What is exponential backoff? One sentence."),
    max_retries=3,
    base_delay=0.5,  # Short delays for demo purposes
)
elapsed = time.perf_counter() - start

print(f"\n✅ Success on attempt {attempts} ({elapsed:.2f}s total)")
md(result)

  ⚠️  Attempt 1 failed: Simulated transient error (attempt 1)
      Retrying in 0.73s (base=0.5s + jitter=0.23s)
  ⚠️  Attempt 2 failed: Simulated transient error (attempt 2)
      Retrying in 1.25s (base=1.0s + jitter=0.25s)

✅ Success on attempt 3 (4.51s total)


Exponential backoff is a network communication strategy that progressively increases the wait time between retries of a failed operation to reduce congestion and improve the chances of success.

The retry logic handled both transient failures automatically. The user never sees the errors — just a slightly delayed response.

### Visualize the Backoff Timing

Let's see how the wait times grow with each attempt.

In [8]:
print("Exponential backoff schedule (base_delay=1.0s, max_delay=30s):")
print(f"{'Attempt':<10} {'Base Delay':<14} {'With Jitter (example)':<22}")
print("-" * 46)

base_delay = 1.0
max_delay = 30.0

for attempt in range(6):
    delay = min(base_delay * (2 ** attempt), max_delay)
    jitter = random.uniform(0, delay * 0.5)
    total = delay + jitter
    bar = "█" * int(total * 2)
    print(f"  {attempt + 1:<8} {delay:<14.1f} {total:<10.2f}s {bar}")

Exponential backoff schedule (base_delay=1.0s, max_delay=30s):
Attempt    Base Delay     With Jitter (example) 
----------------------------------------------
  1        1.0            1.30      s ██
  2        2.0            2.35      s ████
  3        4.0            4.33      s ████████
  4        8.0            8.55      s █████████████████
  5        16.0           23.55     s ███████████████████████████████████████████████
  6        30.0           31.08     s ██████████████████████████████████████████████████████████████


The delays grow exponentially (1 → 2 → 4 → 8 → 16 → 30), giving the server progressively more time to recover. The jitter prevents the **thundering herd** problem where all clients retry simultaneously.

## Step 4 — Fallback Patterns

What happens when **all retries are exhausted** and the call still fails? You have two choices:

1. **Crash** — raise the error (bad user experience)
2. **Fallback** — return a degraded-but-useful response (good user experience)

Common fallback strategies for LLM applications:

| Fallback | Example |
|----------|--------|
| **Cheaper/different model** | Try `gpt-4o-mini` if `gpt-4o` is down |
| **Canned response** | Return a pre-written "I'm having trouble, try again later" message |
| **Cached result** | Serve a stale-but-relevant cached answer (recall from `mini-cache`) |

Let's implement a function that combines retry **and** model fallback.

In [9]:
def chat_with_retry_and_fallback(
    message: str,
    models: list[str] = None,
    max_retries: int = 2,
    base_delay: float = 0.5,
):
    """
    Try each model in order. For each model, retry with exponential backoff.
    If all models fail, return a canned fallback response.

    Returns: dict with response, model_used, attempts, and is_fallback flag.
    """
    if models is None:
        models = ["gpt-4o-mini"]

    for model in models:
        print(f"🔄 Trying model: {model}")
        try:
            def call_llm():
                resp = client.chat.completions.create(
                    model=model,
                    messages=[{"role": "user", "content": message}],
                    temperature=0,
                )
                return resp.choices[0].message.content

            result, attempts = retry_with_backoff(
                func=call_llm,
                max_retries=max_retries,
                base_delay=base_delay,
            )
            return {
                "response": result,
                "model_used": model,
                "attempts": attempts,
                "is_fallback": False,
            }
        except Exception as e:
            print(f"  ❌ All retries exhausted for {model}: {e}")

    # All models failed — return canned fallback
    print("🛟 All models failed. Returning canned fallback response.")
    return {
        "response": "I'm temporarily unable to process your request. Please try again in a moment.",
        "model_used": None,
        "attempts": 0,
        "is_fallback": True,
    }


print("✅ chat_with_retry_and_fallback() defined")

✅ chat_with_retry_and_fallback() defined


### Test 1 — Primary Model Succeeds

When the primary model works, the response comes back normally.

In [10]:
result = chat_with_retry_and_fallback(
    message="What is graceful degradation? One sentence.",
    models=["gpt-4o-mini"],
)

print(f"\nModel used: {result['model_used']}")
print(f"Attempts: {result['attempts']}")
print(f"Fallback: {result['is_fallback']}")
md(result["response"])

🔄 Trying model: gpt-4o-mini

Model used: gpt-4o-mini
Attempts: 1
Fallback: False


Graceful degradation is a design approach that ensures a system continues to function at a reduced level of performance when some components fail or are unavailable, rather than completely breaking down.

### Test 2 — Primary Model Fails, Fallback Model Succeeds

We'll put a non-existent model first, followed by a real one. The function should:
1. Try (and fail) the bad model
2. Fall back to `gpt-4o-mini` and succeed

In [11]:
result = chat_with_retry_and_fallback(
    message="What is a fallback strategy? One sentence.",
    models=["non-existent-model-xyz", "gpt-4o-mini"],
    max_retries=1,     # Fewer retries so the demo runs faster
    base_delay=0.3,
)

print(f"\nModel used: {result['model_used']}")
print(f"Attempts: {result['attempts']}")
print(f"Fallback: {result['is_fallback']}")
md(result["response"])

🔄 Trying model: non-existent-model-xyz
  ⚠️  Attempt 1 failed: Error code: 404 - {'error': {'message': 'The model `non-existent-model-xyz` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'param': None, 'code': 'model_not_found'}}
      Retrying in 0.31s (base=0.3s + jitter=0.01s)
  ⚠️  Attempt 2 failed: Error code: 404 - {'error': {'message': 'The model `non-existent-model-xyz` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'param': None, 'code': 'model_not_found'}}
  ❌ All retries exhausted for non-existent-model-xyz: Error code: 404 - {'error': {'message': 'The model `non-existent-model-xyz` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'param': None, 'code': 'model_not_found'}}
🔄 Trying model: gpt-4o-mini

Model used: gpt-4o-mini
Attempts: 1
Fallback: False


A fallback strategy is a contingency plan designed to be implemented when the primary plan fails or encounters obstacles, ensuring continuity and minimizing disruption.

The system tried the bad model, exhausted retries, then seamlessly fell back to `gpt-4o-mini`. The user gets an answer — they never need to know the primary model was down.

### Test 3 — All Models Fail → Canned Response

When *every* model fails, the function returns a pre-written message instead of crashing.

In [12]:
result = chat_with_retry_and_fallback(
    message="Hello",
    models=["bad-model-1", "bad-model-2"],
    max_retries=1,
    base_delay=0.3,
)

print(f"\nModel used: {result['model_used']}")
print(f"Fallback: {result['is_fallback']}")
md(result["response"])

🔄 Trying model: bad-model-1
  ⚠️  Attempt 1 failed: Error code: 404 - {'error': {'message': 'The model `bad-model-1` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'param': None, 'code': 'model_not_found'}}
      Retrying in 0.34s (base=0.3s + jitter=0.04s)
  ⚠️  Attempt 2 failed: Error code: 404 - {'error': {'message': 'The model `bad-model-1` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'param': None, 'code': 'model_not_found'}}
  ❌ All retries exhausted for bad-model-1: Error code: 404 - {'error': {'message': 'The model `bad-model-1` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'param': None, 'code': 'model_not_found'}}
🔄 Trying model: bad-model-2
  ⚠️  Attempt 1 failed: Error code: 404 - {'error': {'message': 'The model `bad-model-2` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'param': None, 'code': 'model_not_found'}}
      Retrying

I'm temporarily unable to process your request. Please try again in a moment.

Even in the worst case — all models down, all retries exhausted — the user gets a polite message instead of a stack trace. This is **graceful degradation** in action.

## When to Apply Each Pattern

| Scenario | Strategy |
|----------|----------|
| Occasional rate-limit (429) or timeout | **Retry with backoff** — the issue is temporary |
| Primary model is slow or down | **Model fallback** — switch to a cheaper/faster model |
| All external calls fail | **Canned response** — return a helpful static message |
| User can tolerate stale data | **Cached fallback** — serve the last known good response |
| Error is a client bug (400, invalid input) | **Don't retry** — the same request will fail again |

## 🎯 Summary

### Key Takeaways

1. **Exponential backoff** — each retry waits progressively longer (1s → 2s → 4s → ...), giving the server time to recover without being hammered by repeated requests
2. **Jitter** — adding randomness to the wait time prevents multiple clients from retrying at the exact same moment (thundering herd problem)
3. **Model fallback** — when the primary model fails after all retries, switching to an alternative model keeps the application functional
4. **Canned responses** — the last line of defense; when all models fail, a pre-written message is far better than an unhandled exception
5. **Don't retry client errors** — only transient server-side errors (429, 503, timeouts) warrant retries; a 400 Bad Request will fail every time